# 02 — Design and evaluate chunk boundaries

**Level:** Beginner · **Estimated time:** 80–100 minutes · **Scenario:** Harborline Support

You will compare fixed-size, heading-aware, and sentence-window chunking; measure the resulting units; and decide which design preserves the evidence needed for support questions.


## How to use this notebook

Work in this order: read the concept, run the deterministic code, change **one** variable, inspect the trace, and write down what changed. The model API is deliberately absent: the learning objective is to understand the evidence system that an LLM would depend on.

**Scenario.** You are building a small, internal assistant for Harborline, a fictional SaaS company. Support needs trustworthy answers about customer communication and production escalation. The corpus is intentionally tiny so every result can be inspected.


## 1. Chunking is information design

Retrievers do not return “the document”; they return a unit. A chunk boundary decides what can be retrieved, cited, authorized, reranked, and fit into context. The best chunking strategy is therefore empirical: evaluate it against the kinds of questions users actually ask.

```text
source document
   ├─ fixed character windows     → predictable size; can split an idea
   ├─ heading-aware sections      → preserves author structure; can be too large
   └─ sentence windows            → readable fragments; may lose hierarchy
                    ↓
          retrieve + evaluate on question set
```

Chunking should preserve a stable ID, source, and where possible a section, version, location, tenant, and access label.


In [ ]:
from pathlib import Path
from examples.beginner.chunking_lab import (
    answer_coverage, by_heading, by_sentence_window, describe_chunks, fixed_size,
)

source = 'harborline-support'
ROOT = Path.cwd() if (Path.cwd() / 'examples').exists() else Path('../..')
text = (ROOT / 'examples/data/beginner-docs/harborline-support.md').read_text(encoding='utf-8')
print(text)


## 2. Strategy A: fixed-size windows

Fixed-size chunks are a useful baseline. They provide predictable upper bounds for prompt construction and index size, but character boundaries can separate a rule from its qualification. Overlap reduces that risk by repeating text, at a cost: duplicated tokens can crowd context and produce redundant citations.


In [ ]:
fixed = fixed_size(text, source, size=180, overlap=35)
print(describe_chunks(fixed))
for chunk in fixed:
    print(f'\n{chunk.chunk_id}: {chunk.text}')


## 3. Strategy B: heading-aware chunks

Documentation often has meaningful headings. A heading-aware splitter keeps the policy title with its body, which improves citation readability and user review. The trade-off is uneven chunk length: one long section can exceed a context budget or dilute relevance.


In [ ]:
headed = by_heading(text, source)
print(describe_chunks(headed))
for chunk in headed:
    print(f'\n{chunk.chunk_id} [{chunk.section}]: {chunk.text}')


## 4. Strategy C: sentence windows

Sentence windows are a middle ground for narrative text. They avoid cutting a sentence in half and give a controllable overlap. They do not understand tables, code blocks, or document hierarchy; use source-aware parsing when the content has richer structure.


In [ ]:
sentences = by_sentence_window(text, source, sentences_per_chunk=2, overlap_sentences=1)
print(describe_chunks(sentences))
for chunk in sentences:
    print(f'\n{chunk.chunk_id}: {chunk.text}')


## 5. Measure before choosing

Chunk count affects index size and first-stage retrieval cost. Chunk length affects whether enough evidence fits in one hit. Neither metric proves quality by itself, but both help explain results.

| Question shape | Likely helpful boundary | Risk to test |
| --- | --- | --- |
| policy rule + exception | heading or semantic unit | exception split away |
| short FAQ | small sentence window | too much duplicated context |
| long manual section | parent/child hierarchy | child lacks context |
| table or code | structure-aware parser | rows or blocks flattened |


In [ ]:
strategies = {'fixed': fixed, 'heading': headed, 'sentence': sentences}
for name, chunks in strategies.items():
    print(f'{name:8} {describe_chunks(chunks)}')


## 6. Evidence coverage test

Before adding an LLM, ask whether a single chunk holds the key terms required to support a question. This simple diagnostic is not semantic evaluation, but it exposes a common boundary failure: the action appears in one chunk while its approval requirement is in another.


In [ ]:
required = {'restart', 'approval'}
for name, chunks in strategies.items():
    print(name, 'chunks containing all required terms:', answer_coverage(chunks, required))


## 7. Deliberate failure: a boundary that loses a qualification

Use a small fixed window with no overlap. If “restart” and “approval” are split, a generator receiving only the top hit could overstate the policy. The fix is not automatically “make every chunk huge”; compare overlap, heading preservation, parent-document retrieval, and reranking against a test set.


In [ ]:
broken = fixed_size(text, source, size=100, overlap=0)
print(describe_chunks(broken))
for chunk in broken:
    print(chunk.chunk_id, '→', chunk.text)
print('Coverage:', answer_coverage(broken, required))


## 8. Practical design rules

- Match chunking to source structure and question shape; do not use one global setting by habit.
- Preserve document and section metadata so the answer can cite a human-readable location.
- Keep chunk IDs deterministic. A random ID breaks evaluation sets and citation traces.
- Evaluate both a direct question and one whose evidence crosses a boundary.
- Treat overlap as a cost/quality trade-off. Measure redundant context and citation duplication.
- Use specialized parsers for tables, code, and layout-sensitive documents rather than flattening them into prose.


## 9. Experiment: select a policy for Harborline

Create two questions:

1. a direct policy question such as “How often are enterprise customers updated?”
2. a compound question such as “Can support restart a service, and what approval is required?”

For each strategy, record chunk count, whether the answer evidence is intact, and the source/section you would cite. Choose the smallest design that reliably preserves the answer—not the largest context by default.


In [ ]:
# Try a new window size or sentence count. Keep the source text and questions fixed.
candidate = by_sentence_window(text, source, sentences_per_chunk=3, overlap_sentences=1)
print(describe_chunks(candidate))
print('Restart + approval coverage:', answer_coverage(candidate, {'restart', 'approval'}))


## 10. Checkpoint and references

1. What is the difference between a predictable chunk size and a good chunk boundary?
2. Why can overlap improve recall while worsening context cost?
3. Which metadata would let a user navigate from a citation to the original section?
4. When should a table use a structure-aware parser instead of ordinary text chunking?

**Next:** use the chunks as evidence objects, attach structured citations, and make answer/no-answer decisions auditable.

### References

- [RAG lifecycle and chunking guide](../../docs/what-is-rag.md)
- [LlamaIndex ingestion and node parsing concepts](https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/)
- [Unstructured documentation](https://docs.unstructured.io/)
